In [1]:
import pandas as pd
import numpy as np
from pypfopt.expected_returns import mean_historical_return
from pypfopt.discrete_allocation import DiscreteAllocation
from collections import OrderedDict
from pandas import DataFrame
from pandas import Series
from typing import Any
from numpy.typing import NDArray
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.efficient_frontier import EfficientFrontier
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel
from typing import TypedDict
import yfinance as yf

import time
import json
from abc import ABC, abstractmethod

import bidask as ba

In [ ]:
import json
with open("class_mapper.txt",'r') as f:
    stored_dict=json.load(f)
equities=list(stored_dict.keys())
x=yf.download(equities,start="2020-04-09",auto_adjust=True,threads=True,interval="1d",group_by="ticker")
close_returns=x.xs("Close",level=1,axis=1).pct_change().dropna()
df=close_returns.to_parquet("latest_close_returns.parquet")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow import keras
import matplotlib.pyplot as plt

from sklearn import preprocessing
from tensorflow.keras import regularizers
#getting data
df = pd.read_parquet("latest_close_returns.parquet")

#deriving label
def create_sequences(data:pd.DataFrame, window:int)->np.typing.NDArray:
    """data is just time series, we need to make labels

    Args:
        data (pd.DataFrame): time-series data.
        window (int): How many rows or data points in the past are used to learn some pattern before predicting the next data point; for example, if window is 7 days, then the first 7 days are used as features to learn the patterns, the data point at the 8th day is the label, which is the value we want to predict based on the learned pattern in the last 7 days.

    Returns:
        : np.typing.NDArray
    """
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data.iloc[ i:i + window ])
        y.append(data.iloc[ i + window , : ])
    return np.array(X), np.array(y)
X, y = create_sequences(df, 100) # play with window

#splitting; not using train_test_split to avoid shuffling.
split = int(0.85 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

#Preprocessing : Scaling. scaling needs 2D data, not 3D data.After scaling is done, reshape again to 3D, as training need 3D data.

X_train_2D = X_train.reshape(-1, X_train.shape[2])
X_test_2D = X_test.reshape(-1, X_test.shape[2])
scaler=preprocessing.RobustScaler() # play with scaler
y_scaler = preprocessing.RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_2D).reshape(X_train.shape) # fit_tranform only once in training data. transform on the rest of the data
X_test_scaled = scaler.transform(X_test_2D).reshape(X_test.shape)
y_train_scaled = y_scaler.fit_transform(y_train)

#validation data is 20% of the training data
val_split = int(0.8 * len(X_train))
X_tr, X_val = X_train_scaled[:val_split], X_train_scaled[val_split:]
y_tr, y_val = y_train_scaled[:val_split], y_train_scaled[val_split:]



#model definition
model = keras.Sequential()
model.add(keras.layers.LSTM(256,return_sequences=True,kernel_regularizer=regularizers.l2(0.0005),input_shape=(#updated
    X_tr.shape[1], X_tr.shape[2])))
model.add(keras.layers.Dropout(0.2))
model.add(keras.layers.LSTM(512,kernel_regularizer=regularizers.l2(0.0005)))#updated
#model.add(keras.layers.LSTM(100))
model.add(keras.layers.Dropout(0.3))#updated
model.add(keras.layers.Dense(484))
model.compile(
        loss=keras.losses.Huber(delta=1.0), #updated
        optimizer=keras.optimizers.Adam(learning_rate=1e-4), #updated
        metrics=["mae"]#updated
    )



#model_trianing
callback=keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0,
    patience=10,
    verbose=0,
    mode="min",
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0,
)
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=150,
    callbacks=[callback],
    batch_size=128 # updated
)
# 8. Plotting Learning Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(history.history["loss"], label="Train Loss (Huber)")
ax1.plot(history.history["val_loss"], label="Validation Loss (Huber)")
ax1.set_title("Learning Curve (Loss)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid()

ax2.plot(history.history["mae"], label="Train MAE")
ax2.plot(history.history["val_mae"], label="Validation MAE")
ax2.set_title("Learning Curve (MAE)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("MAE")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

# 9. Evaluation
pred_scaled = model.predict(X_test_scaled)
pred = y_scaler.inverse_transform(pred_scaled)    

mse = mean_squared_error(y_test, pred)
mae = mean_absolute_error(y_test, pred)


# 10. Plotting Predictions
time = df.index[-len(y_test):]
plt.figure(figsize=(12, 6))

# Plotting actuals
plt.plot(time, y_test[:, 0], label="Actual", alpha=0.7)

# Plotting predictions (should now show variance instead of a flat line)
plt.plot(time, pred[:, 0], linestyle="--", linewidth=2, label="Predicted")

highlight = int(len(time) * 0.85)
if highlight < len(time):
    plt.axvspan(time[highlight], time[-1], alpha=0.1, color='gray')
    
plt.title("Asset Return Predictions vs Ground Truth (Asset 0)")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"MSE : {mse:.6f}")
print(f"MAE : {mae:.6f}")

from scipy.stats import spearmanr

rank_ic, p_value = spearmanr(
y_test.flatten(),
pred.flatten()
)
    
print(f"Rank IC = {rank_ic:.4f}")

/home/abd/myWorkSpace/FinSight_AI/backend/venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 6s -532947us/step - loss: 0.8861 - mae: 0.6763 - val_loss: 0.8724 - val_mae: 0.6666
Epoch 2/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 374ms/step - loss: 0.8645 - mae: 0.6755 - val_loss: 0.8515 - val_mae: 0.6664
Epoch 3/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 186ms/step - loss: 0.8434 - mae: 0.6747 - val_loss: 0.8312 - val_mae: 0.6663
Epoch 4/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 187ms/step - loss: 0.8229 - mae: 0.6742 - val_loss: 0.8115 - val_mae: 0.6662
Epoch 5/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 187ms/step - loss: 0.8032 - mae: 0.6737 - val_loss: 0.7924 - val_mae: 0.6661
Epoch 6/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 332ms/step - loss: 0.7842 - mae: 0.6732 - val_loss: 0.7741 - val_mae: 0.6660
Epoch 7/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 307ms/step - loss: 0.7659 - mae: 0.6729 - val_loss: 0.7565 - val_mae: 0.6659
Epoch 8/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 455ms/step - loss: 0.7482 - mae: 0.6723 - val_loss: 0.7395 - val_mae: 0.6658
Epoch 9/150
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 185ms/step - loss: 0

In [12]:
from scipy.stats import spearmanr

rank_ic, p_value = spearmanr(
y_test.flatten(),
pred.flatten()
)
    
print(f"Rank IC = {rank_ic:.4f}")

Rank IC = 0.0262
